# NeuroForge — Quick Ablation (the make-or-break experiment)

A minimal, self-contained notebook that answers the one question that decides the
paper: **does the corrector improve accuracy (field MSE / rho_Cd), not just the
residual — and does the DEQ corrector beat the feed-forward one?**

Install -> download/cache AirfRANS -> train all ablation arms -> print a
mean+/-std table. Set a **GPU** runtime (L4 recommended). Run all, top to bottom.


## 1 - GPU check


In [ ]:
!nvidia-smi -L || echo 'No GPU - set Runtime > Change runtime type > GPU'


## 2 - Install


In [ ]:
import os, sys, subprocess
for _v in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'):
    os.environ[_v] = str(os.cpu_count() or 4)

REPO_URL = 'https://github.com/ali-kin4/neuroforge-cfd.git'
REPO_DIR = '/content/neuroforge-cfd'
if not os.path.isdir(REPO_DIR):
    rc = os.system(f'git clone --depth 1 {REPO_URL} {REPO_DIR}')
    if rc != 0:
        raise RuntimeError('Clone failed - set REPO_URL to your repo, or upload it.')
os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[data]'], check=True)
for _p in (os.path.join(REPO_DIR, 'src'), REPO_DIR):
    if _p not in sys.path:
        sys.path.insert(0, _p)

import neuroforge as nf
from benchmarks.ablation import run_ablation
print('NeuroForge', nf.__version__, 'ready')


## 3 - Storage (Drive cache makes re-runs instant)


In [ ]:
USE_DRIVE = True
DATA_ROOT = '/content/airfrans_data'   # raw data on fast LOCAL disk
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/neuroforge_cfd'
else:
    BASE = '/content/neuroforge_runs'
CACHE_DIR = os.path.join(BASE, 'cache')
CKPT_DIR = os.path.join(BASE, 'checkpoints')
for d in (DATA_ROOT, CACHE_DIR, CKPT_DIR):
    os.makedirs(d, exist_ok=True)
print('cache ->', CACHE_DIR)


## 4 - Run the ablation

Fast defaults (1 seed, 40 epochs, 150 sims) for a quick directional answer.
For the **full paper run**, set `seeds=(0, 1, 2)`, `epochs=80`, `n_train=400`
(or use `!python benchmarks/ablation.py --source airfrans --task full --seeds 0 1 2 --cache-dir data/cache`).


In [ ]:
TASK = 'full'   # 'full' (800/200) or 'scarce' (smaller, quicker)
try:
    ABL = run_ablation(
        'airfrans', task=TASK, n_train=150, n_val=80, resolution=128,
        seeds=(0,), epochs=40, corrector_epochs=10,
        width=48, modes=20, n_layers=4, batch_size=6,
        root=DATA_ROOT, cache_dir=CACHE_DIR, download=True,
        device='auto', out_dir=os.path.join(CKPT_DIR, 'results'), verbose=True,
    )
    print()
    print('Saved table ->', os.path.join(CKPT_DIR, 'results', 'ablation.md'))
except Exception:
    import traceback
    err = traceback.format_exc()
    print(err)
    with open(os.path.join(CKPT_DIR, 'last_error.txt'), 'w') as f:
        f.write(err)


## How to read it

- **H1 (corrector helps accuracy):** `backbone + DEQ corrector` should have
  **lower MSE** and **rho_Cd closer to 1** than `backbone`.
- **H2 (residual is a valid trust signal):** `residual_error_spearman` **> 0**.
- **H3 (DEQ >= local):** the DEQ row should match or beat `+ local corrector`.

Send these numbers back to iterate. The full table is on Drive at
`checkpoints/results/ablation.md` + `.csv`.
